In [ ]:
import pandas as pd
import numpy as np
import re
import os

def clean_pokemon_market_data(input_filepath, output_filepath):
    """
    Loads, audits, and cleans the raw Pokémon TCG e-commerce dataset.
    Prepares the data for Google BigQuery ingestion and downstream financial analysis.
    """
    print(f"Loading raw dataset from: {input_filepath}...")

    # Check if file exists to prevent errors
    if not os.path.exists(input_filepath):
        print(f"Error: The file '{input_filepath}' was not found.")
        return

    df = pd.read_csv(input_filepath)

    # ---------------------------------------------------------
    # 1. SCHEMA STANDARDIZATION (BigQuery Compatibility)
    # ---------------------------------------------------------
    print("Step 1: Standardizing column names to snake_case...")
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_', regex=False)
                  .str.replace('[^a-z0-9_]', '', regex=True))

    # ---------------------------------------------------------
    # 2. FINANCIAL DATA PARSING (String to Float Conversion)
    # ---------------------------------------------------------
    print("Step 2: Parsing monetary columns for mathematical operations...")
    # Identify any column related to pricing or cost
    price_cols = [col for col in df.columns if 'price' in col or 'usd' in col or 'cost' in col]

    for col in price_cols:
        if df[col].dtype == 'object':
            # Remove currency symbols and commas, then cast to float
            df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # ---------------------------------------------------------
    # 3. DEEP STRING CLEANING (Handling Hidden Characters & Gaps)
    # ---------------------------------------------------------
    print("Step 3: Performing deep string cleaning on categorical variables...")
    cat_cols = df.select_dtypes(include=['object']).columns

    for col in cat_cols:
        # Replace non-breaking spaces (\xa0) and tabs with a standard space
        df[col] = df[col].str.replace(r'[\xa0\t]+', ' ', regex=True)

        # Squash multiple consecutive spaces into a single standard space
        df[col] = df[col].str.replace(r'\s{2,}', ' ', regex=True)

        # Strip trailing and leading whitespace
        df[col] = df[col].str.strip()

        # Standardize empty strings or 'nan' text as true Pandas Nulls (NaN)
        df[col] = df[col].replace({'nan': np.nan, '': np.nan})

    # ---------------------------------------------------------
    # 4. TEMPORAL DATA CASTING (Time-Series Preparation)
    # ---------------------------------------------------------
    print("Step 4: Formatting date columns to DATETIME objects...")
    date_cols = [col for col in df.columns if 'date' in col or 'year' in col]

    for col in date_cols:
        try:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        except Exception as e:
            pass # Keep as original if parsing fails

    # ---------------------------------------------------------
    # 5. EXPORT FOR DATA WAREHOUSE INGESTION
    # ---------------------------------------------------------
    print(f"Step 5: Exporting cleaned dataset to: {output_filepath}...")
    df.to_csv(output_filepath, index=False)

    print("\n--- DATA QUALITY AUDIT COMPLETED SUCCESSFULLY ---")
    print(f"Final shape of the dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
    print("Dataset is now ready for Google BigQuery ingestion.")

# Execution Block
if __name__ == "__main__":
    import kagglehub
    import os

    # 1. Download/Locate the dataset via Kagglehub
    print("Locating dataset via Kagglehub...")
    path = kagglehub.dataset_download("kanchana1990/e-commerce-pokmon-card-pricing-data")

    # 2. Find the exact CSV file in the downloaded folder
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    INPUT_FILE = os.path.join(path, csv_file)

    # 3. Define output path (saves to current working directory)
    OUTPUT_FILE = 'pokemon_cards_bq_ready.csv'

    # 4. Run the cleaning function
    clean_pokemon_market_data(INPUT_FILE, OUTPUT_FILE)

Locating dataset via Kagglehub...
Using Colab cache for faster access to the 'e-commerce-pokmon-card-pricing-data' dataset.
Loading raw dataset from: /kaggle/input/e-commerce-pokmon-card-pricing-data/pokemon_cards_ultimate_2026.csv...
Step 1: Standardizing column names to snake_case...
Step 2: Parsing monetary columns for mathematical operations...
Step 3: Performing deep string cleaning on categorical variables...
Step 4: Formatting date columns to DATETIME objects...
Step 5: Exporting cleaned dataset to: pokemon_cards_bq_ready.csv...

--- DATA QUALITY AUDIT COMPLETED SUCCESSFULLY ---
Final shape of the dataset: 542 rows, 32 columns.
Dataset is now ready for Google BigQuery ingestion.
